In [ ]:
using CairoMakie;       #Visualización en 3D
using LaTeXStrings;     #Paquetería para utilizar texto en LaTeX en las gráficas
using DelimitedFiles;   #Paquetería para leer y escribir archivos
using FFTW;             #Paquetería para usar transformada de Fourier
using Printf;
using ConcaveHull;      #Para realizar un Concave Hull de los histogramas del g(R)
using StatsBase;        #Para emplear algunas funciones básicas de estadística, como el promedio
using LombScargle;
using LsqFit;           #Paquetería para realizar un ajuste por mínimos cuadrados

DataPath = "Quasiperiodic-Tiles/Global Structural Studies/Data/SI_Fig3-B_SigmaSquare_1D/"; #Ruta para guardar los datos

#Función que "suaviza" una gráfica reduciendo el número de puntos por su promedio (media móvil)
function promedio_m(A, i, m)
    Valores = [A[j][2] for j in (i-m):(i+m)]
    return mean(Valores)
end

In [ ]:
#Diccionario con la relación "Simetría Rotacional" -> "Índice de la tesela que domina la longitud de escala" necesario para visualizar los múltiplos de λ_N
Dict_Indices_Lambda = Dict(
                           5  => 1,
                           7  => 0,
                           9  => 4,
                           11 => 1,
                           13 => 0,
                           15 => 7,
                           17 => 1,
                           19 => 9,
                           21 => 10,
                           23 => 11,
                           25 => 1,
                           27 => 12,
                           29 => 0,
                           31 => 14,
                           33 => 15,
                           35 => 0,
                           37 => 16,
                           39 => 18,
                           41 => 20,
                           43 => 21,
                           45 => 20,
                           47 => 23,
                           49 => 23,
                           51 => 1
                          );

Dict_PCH = Dict(
                5  => 61,
                7  => 20,
                9  => 45,
                11 => 21,
                13 => 22,
                15 => 13,
                17 => 17,
                19 => 17,
                21 => 11,
                23 => 13,
                25 => 16,
                27 => 21,
                29 => 28,
                31 => 17,
                33 => 16,
                35 => 20,
                37 => 20,
                39 => 17,
                41 => 18,
                43 => 18,
                45 => 17,
                47 => 18,
                49 => 16,
                51 => 15
               );

### Generación acumulada de las transformadas de Fourier de $\sigma^2$

In [ ]:
# --- Definición de las características del lienzo y las subgráficas en él ---
Fig = Figure(size = (1800, 800)); #Lienzo en blanco donde se graficara
FFT_Ax = Axis(
              Fig[1, 1],                                                   #Posición en el lienzo donde se realizará la gráfica
              #title = L"N = %$(NSides)",                                  #Título de la gráfica
              xlabel = L"R = 2 \pi / k",                                   #Etiqueta que aparece en el eje horizontal
              ylabel = L"A",                                               #Etiqueta que aparece en el eje vertical
              titlesize = 55,                                              #Tamaño del título
              xlabelsize = 55,                                             #Tamaño de la etiqueta al eje horizontal
              ylabelsize = 55,                                             #Tamaño de la etiqueta al eje vertical
              xticklabelsize = 40,                                         #Tamaño para el eje X
              yticklabelsize = 40,                                         #Tamaño para el eje Y
              xticksize = 25,                                              #Tamaño de los ticks horizontales
              yticksize = 25,                                              #Tamaño de los ticks verticales
              limits = (nothing, nothing),                                 #Límites de la visualización para la gráfica
              xscale = log10,
              yscale = log10
             )
hidespines!(FFT_Ax, :t, :r); #Remueve las líneas de la caja que rodea a la gráfica ':t' = top, ':r' = right
hidedecorations!(
                 FFT_Ax,
                 label = false,           #Se oculta o no las etiquetas a los ejes
                 ticklabels = false,      #Se oculta o no los valores de los ticks de los ejes
                 ticks = false            #Se oculta o no los ticks de los ejes
                )

###################################################################################################################
#                                   Datos del sistema cuasiperiódico 2D-1D
###################################################################################################################
NSides = 5;     #Simetría rotacional del sistema cuasiperiódico en 2D (Antes de la proyección)
Angulo = 0;     #Número del ángulo de la recta 1D a la cual se realiza la proyección de vectores 2D
Radio = 1800;   #Radio de la "vecindad circular" en 1D (Mitad del tamaño de la región cuadrada centrada en el origen)
###################################################################################################################
#                             Datos de los notebooks para la generación de datos
###################################################################################################################
Notebooks = 10;         #Número de Notebooks empleados
Vecindades = Int(1e4);  #Número de Vecindades por Notebook
###################################################################################################################
#                                   Datos para el cálculo de la N(R) y N^2(R)
###################################################################################################################
ΔR = 0.05;
Rango = 0.0:ΔR:Radio;

NSides_Array = [5, 11, 21, 31, 41, 51]; #Simetrías rotacionales a analizar
Paleta_Colores = cgrad(:inferno, length(NSides_Array) + 1, categorical = true); #Definimos el gradiente de colores para la gráfica
λ_N_Array = [];     #Arreglo donde se guardarán las λ_N de cada simetría analizada
Amp_λ_N_Array = []; #Arreglo donde se guardarán las amplitudes originales de cada λ_N
for i in 1:length(NSides_Array)
    NSides = NSides_Array[i]; #Simetría rotacional a analizar
    ###################################################################################################################
    #                                   Lectura de los datos de N(R) y N^2(R)
    ###################################################################################################################
    NR_Acumulado = vec(readdlm(DataPath * "NR_N$(NSides)_ThetaStarVectors$(Angulo)_Acumulados$(Vecindades)_DeltaStep0P05_Radius$(Radio)_Nb1.csv"));
    NR2_Acumulado = vec(readdlm(DataPath * "NR2_N$(NSides)_ThetaStarVectors$(Angulo)_Acumulados$(Vecindades)_DeltaStep0P05_Radius$(Radio)_Nb1.csv"));

    for Nb in 2:Notebooks
        NR_Acumulado .+= vec(readdlm(DataPath * "NR_N$(NSides)_ThetaStarVectors$(Angulo)_Acumulados$(Vecindades)_DeltaStep0P05_Radius$(Radio)_Nb$(Nb).csv"));
        NR2_Acumulado .+= vec(readdlm(DataPath * "NR2_N$(NSides)_ThetaStarVectors$(Angulo)_Acumulados$(Vecindades)_DeltaStep0P05_Radius$(Radio)_Nb$(Nb).csv"));
    end

    #Dividimos entre el número de vecindades acumuladas que se sumaron para obtener los datos
    NR_Acumulado = NR_Acumulado ./ (Vecindades * Notebooks);
    NR2_Acumulado = NR2_Acumulado ./ (Vecindades * Notebooks);
    σ2 = NR2_Acumulado .- (NR_Acumulado.^2);

    Final = 0; #Datos del final a eliminar (por errores en inconsistencias de tamaño en vecindades)
    if NSides == 25
        Final = 50;
    elseif NSides == 29 || NSides == 45
        Final = 170;
    end
    ###################################################################################################################
    #                                Parámetros para realizar la transformada de Fourier    
    ###################################################################################################################
    σ2R = σ2[1:end - Final];    #Seleccionamos el inicio y el final de los datos que queremos analizar
    N = length(σ2R);            #Número de datos
    νs = fftfreq(N, 1/(ΔR));    #Rango de frecuencias que obtiene la transformada de fourier
    n = floor(Int, length(νs)/2);
    X2 = fft(σ2R);              #Transformada de Fourier (está en los complejos, así que hay que normalizar con abs)
    ###################################################################################################################
    #                                            Gráfica de la transformada de Fourier    
    ###################################################################################################################
    # --- Gráfica de los datos de la FFT ---
    lines!(FFT_Ax, 1 ./ νs[2:1:n], (10^(3*(i-1))).*(abs.(X2)[2:1:n]),
           color = (Paleta_Colores[i], 1),
          );
    # --- Datos del máximo de la FFT ---
    A_N = maximum(abs.(X2)[2:1:n]);                     #Máximo valor de la amplitud de la FFT
    I_N = findfirst(x -> x == A_N, abs.(X2)[2:1:n]);    #Índice de la entrada asociada a la máxima amplitud de la FFT
    λ_N = (1 ./ νs[2:1:n])[I_N];                        #Valor de la longitud asociada a la máxima amplitud de la FFT
    push!(λ_N_Array, λ_N);
    push!(Amp_λ_N_Array, A_N);
end
###################################################################################################################
#                                            Gráfica del máximo de la transformada de Fourier    
###################################################################################################################
for i in 1:length(NSides_Array)
      scatter!(FFT_Ax, λ_N_Array[i], (10^(3*(i-1)))*Amp_λ_N_Array[i],
               color = :red,
               markersize = 15
              )
end
###################################################################################################################
#                                            Guardamos las gráficas    
###################################################################################################################
Fig            

### Ajuste lineal a los valores de $\lambda_N$

In [ ]:
NSides_Array = 5:2:51;
Lambda_N = [
            0.72,
            3.50194552529182,
            4.22535211267605,
            4.62724935732647,
            6.49819494584837,
            7.3469387755102,
            7.9295154185022,
            9.375,
            10.4046242774566,
            11.3924050632911,
            12.1452702702702,
            12.6760563380281,
            14.4475806451612,
            14.7540983606557,
            15.7894736842105,
            17.4757281553398,
            16.822429906542,
            18.9473684210526,
            20.4545454545454,
            21.4285714285714,
            21.0764705882352,
            23.3766233766233,
            24,
            25.3521126760563
           ];

@. model(x, p) = p[1] + p[2] * (x)
p0 = [0.5, 0.5]
fit = curve_fit(model, NSides_Array[2:end], Lambda_N[2:end], p0); #El ajuste considera los valores desde 2:end porque el caso N = 5 se considera outlier
K = coef(fit)
println(K)

# --- Definimos el lienzo general y el eje donde se graficará la primera imagen en el lienzo completo ---
Fig = Figure(size = (1800, 800)); #Lienzo en blanco donde se graficara
LambdaN_Ax = Axis(
                  Fig[1, 1],                                                   #Posición en el lienzo donde se realizará la gráfica
                  #title = L"N = %$(NSides)",                                  #Título de la gráfica
                  xlabel = L"N",                                               #Etiqueta que aparece en el eje horizontal
                  ylabel = L"\Lambda_{N}",                                     #Etiqueta que aparece en el eje vertical
                  titlesize = 55,                                              #Tamaño del título
                  xlabelsize = 55,                                             #Tamaño de la etiqueta al eje horizontal
                  ylabelsize = 55,                                             #Tamaño de la etiqueta al eje vertical
                  xticklabelsize = 40,                                         #Tamaño para el eje X
                  yticklabelsize = 40,                                         #Tamaño para el eje Y
                  xticksize = 25,                                              #Tamaño de los ticks horizontales
                  yticksize = 25,                                              #Tamaño de los ticks verticales
                  limits = (nothing, nothing),                                 #Límites de la visualización para la gráfica
                  #xscale = log10,
                  #yscale = log10
                 );
hidespines!(LambdaN_Ax, :t, :r);      #Remueve las líneas de la caja que rodea a la gráfica ':t' = top, ':r' = right
hidedecorations!(
                 LambdaN_Ax,
                 label = false,           #Se oculta o no las etiquetas a los ejes
                 ticklabels = false,      #Se oculta o no los valores de los ticks de los ejes
                 ticks = false            #Se oculta o no los ticks de los ejes
                );

lines!(
       LambdaN_Ax, NSides_Array[2:end], K[1] .+ (K[2] .* (NSides_Array[2:end])),
       label = L"\Lambda_{\infty} = %$(round(K[1], digits = 3)) + %$(round(K[2], digits = 3)) N"
      );
scatter!(
         LambdaN_Ax, NSides_Array, Lambda_N,
         markersize = 10,
         color = :red
        )

Fig